In [3]:
import asyncio
tokens_used = 0

async def record_usage(amount):
    global tokens_used

    current = tokens_used
    await asyncio.sleep(0)
    tokens_used = current + amount

async def main():
    global tokens_used

    await asyncio.gather(
        record_usage(100),
        record_usage(200),
    )

    print(tokens_used)

await main()

200


In [7]:
import asyncio

class UsageCounter:
    def __init__(self):
        self.tokens_used=0
        self._lock=asyncio.Lock()

    async def record(self,amount:int):
        async with self._lock:
            current=self.tokens_used

            await asyncio.sleep(0)
            self.tokens_used=current + amount

async def main():
    counter = UsageCounter()

    await asyncio.gather(
        counter.record(100),
        counter.record(200),
    )

    print(counter.tokens_used)


await main()

300


In [ ]:
from dataclasses import dataclass
import math

@dataclass(frozen=True)
class AdmissionDecision:
    accepted: bool
    estimated_completion: float
    reason: str

class AdmissionController:
    def __init__(
            self, workers:int,service_time:float,
    ):
        if workers<=0 or service_time<=0:
            raise ValueError("Invalid Configuration")
        self.workers=workers
        self.service_time=service_time

    def evaluate(
            self, pending_jobs:int,deadline_seconds:float,
    )-> AdmissionDecision:
        if pending_jobs<0:
            raise ValueError("Pending_jobs cannot be negative")
        if deadline_seconds<=0:
            return AdmissionDecision(
                False,0.0,"Deadline expired"
            )

        batches_ahead=pending_jobs//self.workers

        estimated_wait=(
            batches_ahead*self.service_time
        )

        estimated_completion=(
            estimated_wait+self.service_time
        )

        accepted=estimated_completion<=deadline_seconds

        return AdmissionDecision(
            accepted=accepted,
            estimated_completion=estimated_completion,
            reason=(
                "Within estimated deadline"
                if accepted
                else "Predicted deadline miss"
            ),
        )

controller = AdmissionController(
    workers=5,
    service_time=2.0,
)

decision = controller.evaluate(
    pending_jobs=20,
    deadline_seconds=3.0,
)

print(decision)

AttributeError: 'AdmissionController' object has no attribute 'workers'